In [0]:
spark

SparkSession - hive 
 
 
 SparkContext 

 Spark UI 

 
 Version 
 v3.3.2 
 Master 
 local[8] 
 AppName 
 Databricks Shell

**What Happens when we encounter with corrupted record in different read mode?**

- For Permissive mode it will print all the rows but it will set the null value to all corrupted fields

In [0]:
Employee_Df=spark.read.format("csv")\
                .option("header","true")\
                .option("inferschema","false")\
                .option("mode","PERMISSIVE")\
                .load("/FileStore/tables/Employee_data.csv")
Employee_Df.show()

+---+--------+---+------+------------+--------+
| id|    name|age|salary|     address| nominee|
+---+--------+---+------+------------+--------+
|  1|  Manish| 26| 75000|       bihar|nominee1|
|  2|  Nikita| 23|100000|uttarpradesh|nominee2|
|  3|  Pritam| 22|150000|   Bangalore|   India|
|  4|Prantosh| 17|200000|     Kolkata|   India|
|  5|  Vikash| 31|300000|        null|nominee5|
+---+--------+---+------+------------+--------+



- In DROPMALFORMED it will drop all the corrupted records.

In [0]:
Employee_Df1=spark.read.format("csv")\
                .option("header","true")\
                .option("inferschema","false")\
                .option("mode","DROPMALFORMED")\
                .load("/FileStore/tables/Employee_data.csv")
Employee_Df1.show()

+---+------+---+------+------------+--------+
| id|  name|age|salary|     address| nominee|
+---+------+---+------+------------+--------+
|  1|Manish| 26| 75000|       bihar|nominee1|
|  2|Nikita| 23|100000|uttarpradesh|nominee2|
|  5|Vikash| 31|300000|        null|nominee5|
+---+------+---+------+------------+--------+



- In FAILFAST mode it will fail:

In [0]:
Employee_Df2=spark.read.format("csv")\
                .option("header","true")\
                .option("inferschema","true")\
                .option("mode","FAILFAST")\
                .load("/FileStore/tables/Employee_data.csv")



In [0]:
Employee_Df2.show()

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-2965880270080747>:1
----> 1 Employee_Df2.show()

File /databricks/spark/python/pyspark/instrumentation_utils.py:48, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     46 start = time.perf_counter()
     47 try:
---> 48     res = func(*args, **kwargs)
     49     logger.log_success(
     50         module_name, class_name, function_name, time.perf_counter() - start, signature
     51     )
     52     return res

File /databricks/spark/python/pyspark/sql/dataframe.py:920, in DataFrame.show(self, n, truncate, vertical)
    914     raise PySparkTypeError(
    915         error_class="NOT_A_BOOLEAN",
    916         message_parameters={"arg_name": "vertical", "arg_type": type(vertical).__name__},
    917     )
    919 if isinstance(truncate, bool) and truncate:
--> 920     print(self._jdf.showString(n, 20, vertical))
   

**How we print bad records:**

In [0]:
#How we print corrupted records
from pyspark.sql.types import StringType,StructType,StructField,IntegerType
emp_schema=StructType(
                        [
                            StructField("id",IntegerType(),True),
                            StructField("name",StringType(),True),
                            StructField("age",IntegerType(),True),
                            StructField("Salary",IntegerType(),True),
                            StructField("address",StringType(),True),
                            StructField("nominee",StringType(),True),
                            StructField("_corrupt_record",StringType(),True)
                        ])

In [0]:
Employee_Df3 = spark.read.format("csv") \
                .option("header","true") \
                .option("inferschema","true")\
                .option("mode","PERMISSIVE") \
                .schema(emp_schema)\
                .load("/FileStore/tables/Employee_data.csv")
Employee_Df3.show()

+---+--------+---+------+------------+--------+--------------------+
| id|    name|age|Salary|     address| nominee|     _corrupt_record|
+---+--------+---+------+------------+--------+--------------------+
|  1|  Manish| 26| 75000|       bihar|nominee1|                null|
|  2|  Nikita| 23|100000|uttarpradesh|nominee2|                null|
|  3|  Pritam| 22|150000|   Bangalore|   India|3,Pritam,22,15000...|
|  4|Prantosh| 17|200000|     Kolkata|   India|4,Prantosh,17,200...|
|  5|  Vikash| 31|300000|        null|nominee5|                null|
+---+--------+---+------+------------+--------+--------------------+



In [0]:
Employee_Df3.show(truncate=False)

+---+--------+---+------+------------+--------+-------------------------------------------+
|id |name    |age|Salary|address     |nominee |_corrupt_record                            |
+---+--------+---+------+------------+--------+-------------------------------------------+
|1  |Manish  |26 |75000 |bihar       |nominee1|null                                       |
|2  |Nikita  |23 |100000|uttarpradesh|nominee2|null                                       |
|3  |Pritam  |22 |150000|Bangalore   |India   |3,Pritam,22,150000,Bangalore,India,nominee3|
|4  |Prantosh|17 |200000|Kolkata     |India   |4,Prantosh,17,200000,Kolkata,India,nominee4|
|5  |Vikash  |31 |300000|null        |nominee5|null                                       |
+---+--------+---+------+------------+--------+-------------------------------------------+



**Where do we store bad records?**

In [0]:
Employee_Df5=spark.read.format("csv")\
                .option("header","true")\
                .option("inferschema","false")\
                .schema(emp_schema)\
                .option("badRecordsPath","/FileStore/tables/bad_Records_Employee")\
                .load("/FileStore/tables/Employee_data.csv")
Employee_Df5.show(truncate=False)

+---+------+---+------+------------+--------+---------------+
|id |name  |age|Salary|address     |nominee |_corrupt_record|
+---+------+---+------+------------+--------+---------------+
|1  |Manish|26 |75000 |bihar       |nominee1|null           |
|2  |Nikita|23 |100000|uttarpradesh|nominee2|null           |
|5  |Vikash|31 |300000|null        |nominee5|null           |
+---+------+---+------+------------+--------+---------------+



In [0]:
%fs
ls /FileStore/tables/bad_Records_Employee/20250629T151929/bad_records/

path,name,size,modificationTime
dbfs:/FileStore/tables/bad_Records_Employee/20250629T151929/bad_records/part-00000-66f44acb-a336-40ec-b844-b47b6fe6aa95,part-00000-66f44acb-a336-40ec-b844-b47b6fe6aa95,494,1751210371000


In [0]:
bad_data_df=spark.read.format("json").load("/FileStore/tables/bad_Records_Employee/20250629T151929/bad_records/")
bad_data_df.show(truncate=False)

+----------------------------------------+--------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------+
|path                                    |reason                                                                                                                          |record                                     |
+----------------------------------------+--------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------+
|dbfs:/FileStore/tables/Employee_data.csv|org.apache.spark.SparkRuntimeException: [MALFORMED_CSV_RECORD] Malformed CSV record: 3,Pritam,22,150000,Bangalore,India,nominee3|3,Pritam,22,150000,Bangalore,India,nominee3|
|dbfs:/FileStore/tables/Employee_data.csv|org.apache.spark.SparkRuntimeException: [MALFORMED_CSV_RECORD] Malformed CSV record: 4,Prantos